# CicloAlerta: treino do detector de buracos e rachaduras

Este notebook treina o YOLO11n com a base pública RDD2022, avalia o modelo em imagens
que ele nunca viu e gera o arquivo `resultados.zip`.

**Como usar:** menu **Ambiente de execução > Executar tudo**. Mantenha a aba aberta até o
fim. No final, o `resultados.zip` é baixado sozinho. Se o download não começar, abra o
ícone de pasta na lateral, procure `cicloalerta/resultados.zip` e baixe por lá.

O notebook já pede GPU T4. Se a primeira célula acusar falta de GPU, vá em
**Ambiente de execução > Alterar o tipo de ambiente de execução > GPU T4** e rode de novo.

## 1. Ambiente

In [ ]:
import os
import shutil
import subprocess
import sys

if shutil.which("nvidia-smi") is None or subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    raise RuntimeError(
        "Sem GPU. Vá em Ambiente de execução > Alterar o tipo de ambiente de execução > GPU T4 e rode de novo."
    )

os.environ["WANDB_MODE"] = "disabled"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.200", "onnx", "onnxslim", "onnxruntime"],
    check=True,
)
print("Ambiente pronto.")

## 2. Configuração

In [ ]:
from pathlib import Path

PAISES = ["India", "Japan", "Czech"]  # captura com celular em carro, como no uso previsto
LIMITE_POR_PAIS = 2500                # imagens com defeito por país, para o treino caber em cerca de 1 hora
FRACAO_FUNDO = 0.10                   # imagens sem defeito, que ensinam o modelo a ficar quieto
EPOCAS = 50
PACIENCIA = 12                        # para antes se a validação não melhorar por 12 épocas
TAMANHO = 640
LOTE = 32
MODELO_BASE = "yolo11n.pt"            # pré-treinado no COCO, ponto de partida do fine-tuning
SEMENTE = 42

RAIZ = Path.cwd() / "cicloalerta"
ORIGEM = RAIZ / "rdd2022"
DESTINO = RAIZ / "via_yolo"
EXECUCOES = RAIZ / "execucoes"
ORIGEM.mkdir(parents=True, exist_ok=True)
EXECUCOES.mkdir(parents=True, exist_ok=True)

## 3. Código do projeto

Os dois arquivos abaixo são cópias de `cicloalerta/` no repositório.

In [ ]:
%%writefile preparar_dados.py
#!/usr/bin/env python3
"""Converte o RDD2022 (Pascal VOC) para YOLO, com duas classes: buraco e rachadura.

As imagens do RDD2022 vêm de veículos em movimento, então quadros próximos podem
ser quase iguais. Para isso não vazar entre treino e teste, a divisão é feita por
blocos de nomes consecutivos, e não imagem a imagem. A divisão usa hash, logo é a
mesma em qualquer máquina.

Uso:
    python preparar_dados.py --origem dados/rdd2022 --destino dados/via_yolo
"""
from __future__ import annotations

import argparse
import hashlib
import json
import os
import re
import shutil
import sys
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path

CLASSES = ("buraco", "rachadura")
ID_BURACO = 0
ID_RACHADURA = 1

# D00, D10 e D20 descrevem o formato da trinca. Na triagem, o que pesa é separar
# trinca de buraco, e juntar as três categorias dá mais exemplos ao modelo.
MAPA_ROTULOS = {"D40": ID_BURACO, "D00": ID_RACHADURA, "D10": ID_RACHADURA, "D20": ID_RACHADURA}

TAMANHO_BLOCO = 50
PARTICOES = ("train", "val", "test")
_PADRAO_NOME = re.compile(r"^(?P<prefixo>.*?)_?(?P<numero>\d+)$")


@dataclass
class Amostra:
    """Uma imagem anotada, com as caixas já no formato YOLO."""

    nome: str
    pais: str
    imagem: Path
    caixas: list[tuple[int, float, float, float, float]] = field(default_factory=list)

    def tem_classe(self, classe_id: int) -> bool:
        return any(caixa[0] == classe_id for caixa in self.caixas)


def ler_anotacao(caminho_xml: Path) -> tuple[int, int, list[tuple[str, float, float, float, float]]]:
    """Lê largura, altura e os objetos (rótulo, xmin, ymin, xmax, ymax) de um XML VOC."""
    raiz = ET.parse(caminho_xml).getroot()
    tamanho = raiz.find("size")
    largura = int(float(tamanho.findtext("width", "0"))) if tamanho is not None else 0
    altura = int(float(tamanho.findtext("height", "0"))) if tamanho is not None else 0
    objetos = []
    for objeto in raiz.iter("object"):
        rotulo = (objeto.findtext("name") or "").strip()
        caixa = objeto.find("bndbox")
        if caixa is None or not rotulo:
            continue
        try:
            xmin, ymin, xmax, ymax = (float(caixa.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax"))
        except (TypeError, ValueError):
            continue
        objetos.append((rotulo, xmin, ymin, xmax, ymax))
    return largura, altura, objetos


def para_yolo(
    xmin: float, ymin: float, xmax: float, ymax: float, largura: int, altura: int
) -> tuple[float, float, float, float] | None:
    """Converte para (cx, cy, l, a) normalizados. Devolve None se a caixa for inválida."""
    x1, x2 = sorted((max(0.0, min(xmin, largura)), max(0.0, min(xmax, largura))))
    y1, y2 = sorted((max(0.0, min(ymin, altura)), max(0.0, min(ymax, altura))))
    if x2 - x1 < 2 or y2 - y1 < 2:
        return None
    return (
        (x1 + x2) / 2 / largura,
        (y1 + y2) / 2 / altura,
        (x2 - x1) / largura,
        (y2 - y1) / altura,
    )


def hash_estavel(texto: str) -> int:
    return int(hashlib.sha1(texto.encode("utf-8")).hexdigest(), 16)


def bloco_da_imagem(nome: str) -> str:
    """Agrupa nomes consecutivos: India_000123 e India_000149 caem no mesmo bloco."""
    achado = _PADRAO_NOME.match(nome)
    if achado is None:
        return nome
    return f"{achado['prefixo']}#{int(achado['numero']) // TAMANHO_BLOCO}"


def particao_do_bloco(pais: str, bloco: str, frac_val: float = 0.1, frac_teste: float = 0.1) -> str:
    valor = hash_estavel(f"{pais}:{bloco}") % 10_000 / 10_000
    if valor < frac_teste:
        return "test"
    if valor < frac_teste + frac_val:
        return "val"
    return "train"


def _pasta_de_treino(caminho_xml: Path) -> Path | None:
    """Sobe na árvore até a pasta train, que define o país e onde estão as imagens."""
    return next((pasta for pasta in caminho_xml.parents if pasta.name == "train"), None)


def coletar_amostras(origem: Path, estatisticas: dict) -> list[Amostra]:
    """Acha os XML sob <pais>/train, aceitando annotations/ ou annotations/xmls/."""
    amostras = []
    for caminho_xml in sorted(origem.rglob("*.xml")):
        pasta_treino = _pasta_de_treino(caminho_xml)
        if pasta_treino is None:
            continue
        pais = pasta_treino.parent.name
        imagem = pasta_treino / "images" / f"{caminho_xml.stem}.jpg"
        if not imagem.exists():
            estatisticas["descartes"]["imagem_ausente"] += 1
            continue
        try:
            largura, altura, objetos = ler_anotacao(caminho_xml)
        except ET.ParseError:
            estatisticas["descartes"]["xml_invalido"] += 1
            continue
        if largura <= 0 or altura <= 0:
            estatisticas["descartes"]["tamanho_ausente"] += 1
            continue
        amostra = Amostra(caminho_xml.stem, pais, imagem)
        for rotulo, xmin, ymin, xmax, ymax in objetos:
            if rotulo not in MAPA_ROTULOS:
                estatisticas["rotulos_ignorados"][rotulo] += 1
                continue
            caixa = para_yolo(xmin, ymin, xmax, ymax, largura, altura)
            if caixa is None:
                estatisticas["descartes"]["caixa_invalida"] += 1
                continue
            estatisticas["rotulos_originais"][rotulo] += 1
            amostra.caixas.append((MAPA_ROTULOS[rotulo], *caixa))
        amostras.append(amostra)
    return amostras


def selecionar(amostras: list[Amostra], limite: int | None, frac_fundo: float) -> list[Amostra]:
    """Limita um país priorizando buracos, a classe mais rara, e guarda um pouco de fundo."""
    ordenadas = sorted(amostras, key=lambda a: hash_estavel(a.nome))
    com_buraco = [a for a in ordenadas if a.tem_classe(ID_BURACO)]
    so_rachadura = [a for a in ordenadas if a.caixas and not a.tem_classe(ID_BURACO)]
    fundo = [a for a in ordenadas if not a.caixas]
    positivas = com_buraco + so_rachadura
    if limite is not None:
        positivas = positivas[:limite]
    quantidade_fundo = round(len(positivas) * frac_fundo / (1 - frac_fundo))
    return positivas + fundo[:quantidade_fundo]


def _ligar(origem: Path, destino: Path) -> None:
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        destino.unlink()
    try:
        os.link(origem, destino)
    except OSError:
        shutil.copy2(origem, destino)


def _rotulo_em_texto(amostra: Amostra) -> str:
    linhas = [f"{c} {cx:.6f} {cy:.6f} {l:.6f} {a:.6f}" for c, cx, cy, l, a in amostra.caixas]
    return "\n".join(linhas) + ("\n" if linhas else "")


def _yaml(pasta: Path, treino: str, validacao: str, teste: str) -> str:
    nomes = "\n".join(f"  {i}: {nome}" for i, nome in enumerate(CLASSES))
    return (
        f"path: {pasta.resolve().as_posix()}\n"
        f"train: {treino}\nval: {validacao}\ntest: {teste}\n"
        f"names:\n{nomes}\n"
    )


def _novo_contador() -> dict:
    return {"imagens": 0, "fundo": 0, **{classe: 0 for classe in CLASSES}}


def _somar(contador: dict, amostra: Amostra) -> None:
    contador["imagens"] += 1
    contador["fundo"] += 0 if amostra.caixas else 1
    for caixa in amostra.caixas:
        contador[CLASSES[caixa[0]]] += 1


def preparar(
    origem: Path,
    destino: Path,
    limite_por_pais: int | None = None,
    frac_fundo: float = 0.1,
    frac_val: float = 0.1,
    frac_teste: float = 0.1,
) -> dict:
    estatisticas: dict = {
        "classes": list(CLASSES),
        "mapa_rotulos": MAPA_ROTULOS,
        "tamanho_bloco": TAMANHO_BLOCO,
        "rotulos_originais": Counter(),
        "rotulos_ignorados": Counter(),
        "descartes": Counter(),
    }
    amostras = coletar_amostras(origem, estatisticas)
    if not amostras:
        raise SystemExit(f"Nenhuma anotação encontrada em {origem}. Confira a extração dos arquivos.")

    if destino.exists():
        shutil.rmtree(destino)

    por_pais: dict[str, list[Amostra]] = defaultdict(list)
    for amostra in amostras:
        por_pais[amostra.pais].append(amostra)

    contagem_particao = {p: _novo_contador() for p in PARTICOES}
    contagem_pais: dict[str, dict] = {}
    teste_por_pais: dict[str, list[Amostra]] = defaultdict(list)

    for pais in sorted(por_pais):
        contagem_pais[pais] = {"disponiveis": len(por_pais[pais]), **{p: _novo_contador() for p in PARTICOES}}
        for amostra in selecionar(por_pais[pais], limite_por_pais, frac_fundo):
            bloco = bloco_da_imagem(amostra.nome)
            particao = particao_do_bloco(amostra.pais, bloco, frac_val, frac_teste)
            _ligar(amostra.imagem, destino / "images" / particao / f"{amostra.nome}.jpg")
            rotulo = destino / "labels" / particao / f"{amostra.nome}.txt"
            rotulo.parent.mkdir(parents=True, exist_ok=True)
            rotulo.write_text(_rotulo_em_texto(amostra), encoding="utf-8")
            _somar(contagem_particao[particao], amostra)
            _somar(contagem_pais[pais][particao], amostra)
            if particao == "test":
                teste_por_pais[pais].append(amostra)

    (destino / "dados.yaml").write_text(
        _yaml(destino, "images/train", "images/val", "images/test"), encoding="utf-8"
    )

    # Uma pasta de teste por país, para medir se o modelo vai pior em algum deles.
    # O YOLO exige as chaves train e val no YAML, então elas apontam para o próprio teste.
    for pais, lista in teste_por_pais.items():
        pasta = destino / "paises" / pais
        for amostra in lista:
            _ligar(amostra.imagem, pasta / "images" / "test" / f"{amostra.nome}.jpg")
            rotulo = pasta / "labels" / "test" / f"{amostra.nome}.txt"
            rotulo.parent.mkdir(parents=True, exist_ok=True)
            rotulo.write_text(_rotulo_em_texto(amostra), encoding="utf-8")
        (pasta / "dados.yaml").write_text(
            _yaml(pasta, "images/test", "images/test", "images/test"), encoding="utf-8"
        )

    estatisticas.update(particoes=contagem_particao, por_pais=contagem_pais)
    for chave in ("rotulos_originais", "rotulos_ignorados", "descartes"):
        estatisticas[chave] = dict(sorted(estatisticas[chave].items()))
    (destino / "estatisticas.json").write_text(
        json.dumps(estatisticas, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return estatisticas


def main() -> int:
    analisador = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    analisador.add_argument("--origem", type=Path, required=True, help="pasta com os países do RDD2022 extraídos")
    analisador.add_argument("--destino", type=Path, required=True, help="pasta de saída no formato YOLO")
    analisador.add_argument("--limite-por-pais", type=int, default=None, help="máximo de imagens com defeito por país")
    analisador.add_argument("--frac-fundo", type=float, default=0.1, help="fração de imagens sem defeito")
    args = analisador.parse_args()

    estatisticas = preparar(args.origem, args.destino, args.limite_por_pais, args.frac_fundo)
    print(json.dumps(estatisticas["particoes"], ensure_ascii=False, indent=2))
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile detector.py
"""Inferência do modelo exportado para ONNX, sem depender de PyTorch.

Espera a saída padrão do YOLO11 exportado: (1, 4 + classes, N), com caixas em
(cx, cy, l, a), em pixels da entrada, e pontuações já passadas pela sigmoide.
"""
from __future__ import annotations

import ast
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Mapping, Sequence

import cv2
import numpy as np

NOMES_PADRAO = {0: "buraco", 1: "rachadura"}


@dataclass(frozen=True)
class Deteccao:
    classe_id: int
    classe: str
    confianca: float
    caixa: tuple[float, float, float, float]  # x1, y1, x2, y2 em pixels da imagem original


@dataclass(frozen=True)
class Ajuste:
    """Como a imagem original foi encaixada na entrada do modelo."""

    escala: float
    margem_x: int
    margem_y: int


def letterbox(imagem: np.ndarray, tamanho: tuple[int, int]) -> tuple[np.ndarray, Ajuste]:
    """Redimensiona sem distorcer e completa com cinza, como no treino."""
    altura, largura = imagem.shape[:2]
    alvo_a, alvo_l = tamanho
    escala = min(alvo_a / altura, alvo_l / largura)
    nova_l, nova_a = round(largura * escala), round(altura * escala)
    if (nova_l, nova_a) != (largura, altura):
        imagem = cv2.resize(imagem, (nova_l, nova_a), interpolation=cv2.INTER_LINEAR)
    margem_y = round((alvo_a - nova_a) / 2 - 0.1)
    margem_x = round((alvo_l - nova_l) / 2 - 0.1)
    saida = np.full((alvo_a, alvo_l, 3), 114, dtype=np.uint8)
    saida[margem_y : margem_y + nova_a, margem_x : margem_x + nova_l] = imagem
    return saida, Ajuste(escala, margem_x, margem_y)


def nms(caixas: np.ndarray, pontuacoes: np.ndarray, limiar_iou: float) -> list[int]:
    """Supressão não máxima. Devolve os índices mantidos, do mais confiante ao menos."""
    ordem = pontuacoes.argsort()[::-1]
    areas = (caixas[:, 2] - caixas[:, 0]).clip(0) * (caixas[:, 3] - caixas[:, 1]).clip(0)
    mantidos: list[int] = []
    while ordem.size:
        atual = int(ordem[0])
        mantidos.append(atual)
        resto = ordem[1:]
        x1 = np.maximum(caixas[atual, 0], caixas[resto, 0])
        y1 = np.maximum(caixas[atual, 1], caixas[resto, 1])
        x2 = np.minimum(caixas[atual, 2], caixas[resto, 2])
        y2 = np.minimum(caixas[atual, 3], caixas[resto, 3])
        intersecao = (x2 - x1).clip(0) * (y2 - y1).clip(0)
        uniao = areas[atual] + areas[resto] - intersecao
        iou = np.divide(intersecao, uniao, out=np.zeros_like(intersecao, dtype=float), where=uniao > 0)
        ordem = resto[iou <= limiar_iou]
    return mantidos


def decodificar(
    saida: np.ndarray,
    ajuste: Ajuste,
    tamanho_original: tuple[int, int],
    nomes: Mapping[int, str],
    limiares: np.ndarray,
    limiar_iou: float = 0.5,
    maximo: int = 100,
) -> list[Deteccao]:
    """Transforma a saída bruta da rede em detecções na escala da imagem original."""
    if saida.ndim != 3 or saida.shape[0] != 1 or saida.shape[1] != 4 + len(nomes):
        raise ValueError(
            f"Saída {saida.shape} não é (1, {4 + len(nomes)}, N). "
            "Exporte um modelo YOLO11 para ONNX sem NMS embutido."
        )
    predicoes = saida[0].T
    pontuacoes_classes = predicoes[:, 4:]
    classes = pontuacoes_classes.argmax(1)
    confiancas = pontuacoes_classes[np.arange(len(classes)), classes]
    mascara = confiancas >= limiares[classes]
    if not mascara.any():
        return []

    cx, cy, largura, altura = predicoes[mascara, :4].T
    caixas = np.stack((cx - largura / 2, cy - altura / 2, cx + largura / 2, cy + altura / 2), axis=1)
    classes, confiancas = classes[mascara], confiancas[mascara]

    # A soma por classe separa as caixas no espaço, então o NMS atua dentro de cada classe.
    deslocamento = classes[:, None].astype(caixas.dtype) * 10_000.0
    mantidos = nms(caixas + deslocamento, confiancas, limiar_iou)[:maximo]

    altura_img, largura_img = tamanho_original
    caixas[:, [0, 2]] = ((caixas[:, [0, 2]] - ajuste.margem_x) / ajuste.escala).clip(0, largura_img)
    caixas[:, [1, 3]] = ((caixas[:, [1, 3]] - ajuste.margem_y) / ajuste.escala).clip(0, altura_img)

    return [
        Deteccao(
            classe_id=int(classes[i]),
            classe=nomes[int(classes[i])],
            confianca=float(confiancas[i]),
            caixa=tuple(float(v) for v in caixas[i]),
        )
        for i in mantidos
    ]


def _ler_metadados(sessao) -> tuple[dict[int, str] | None, tuple[int, int] | None]:
    metadados = sessao.get_modelmeta().custom_metadata_map
    nomes = imgsz = None
    if "names" in metadados:
        nomes = {int(k): str(v) for k, v in ast.literal_eval(metadados["names"]).items()}
    if "imgsz" in metadados:
        valor = ast.literal_eval(metadados["imgsz"])
        imgsz = (int(valor[0]), int(valor[1])) if isinstance(valor, (list, tuple)) else (int(valor), int(valor))
    return nomes, imgsz


class DetectorONNX:
    """Carrega o modelo e detecta em imagens BGR, o formato usado pelo OpenCV."""

    def __init__(
        self,
        modelo: str | Path,
        limiares: Mapping[str, float] | None = None,
        limiar_padrao: float = 0.25,
        limiar_iou: float = 0.5,
        provedores: Sequence[str] = ("CPUExecutionProvider",),
        sessao=None,
    ) -> None:
        if sessao is None:
            import onnxruntime as ort

            caminho = Path(modelo)
            if not caminho.exists():
                raise FileNotFoundError(f"Modelo não encontrado: {caminho}. Rode o notebook de treino primeiro.")
            sessao = ort.InferenceSession(str(caminho), providers=list(provedores))
            if limiares is None:
                arquivo = caminho.with_name("limiares.json")
                if arquivo.exists():
                    limiares = json.loads(arquivo.read_text(encoding="utf-8"))

        self.sessao = sessao
        self.entrada = sessao.get_inputs()[0].name
        nomes, imgsz = _ler_metadados(sessao)
        self.nomes = nomes or NOMES_PADRAO
        self.imgsz = imgsz or (640, 640)
        self.limiar_iou = limiar_iou
        self.limiares = np.full(len(self.nomes), limiar_padrao, dtype=np.float32)
        for classe_id, nome in self.nomes.items():
            if limiares and nome in limiares:
                self.limiares[classe_id] = limiares[nome]

    def preprocessar(self, quadro: np.ndarray) -> tuple[np.ndarray, Ajuste]:
        encaixada, ajuste = letterbox(quadro, self.imgsz)
        tensor = encaixada[:, :, ::-1].transpose(2, 0, 1)[None].astype(np.float32) / 255.0
        return tensor, ajuste

    def detectar(self, quadro: np.ndarray) -> list[Deteccao]:
        tensor, ajuste = self.preprocessar(quadro)
        saida = self.sessao.run(None, {self.entrada: tensor})[0]
        return decodificar(saida, ajuste, quadro.shape[:2], self.nomes, self.limiares, self.limiar_iou)

## 4. Baixar a base RDD2022

Os links oficiais responderam 403 em 04/11/2026, então usamos dois espelhos no Hugging Face.
As imagens e as trincas (D00, D10 e D20) vêm de `dronefreak/RDD2022`. A classe 3 desse espelho
marca faixas e sinalização, e não buraco, por isso os buracos (D40) vêm de `edwrdc/RDD2022`.
Só entram imagens presentes nos dois. Licença da base: CC BY-SA 4.0.

In [ ]:
import concurrent.futures
import json
import re
import time
import urllib.error
import urllib.request
from collections import defaultdict

import cv2
import numpy as np

import preparar_dados

REPO_IMAGENS = "dronefreak/RDD2022"
REVISAO_IMAGENS = "72b01f94c2e5bfe430861afd73995f2d412b8855"
REPO_BURACOS = "edwrdc/RDD2022"
REVISAO_BURACOS = "d89672227027116a551aa3bfcc69efdf658b9579"
BASE_IMAGENS = f"https://huggingface.co/datasets/{REPO_IMAGENS}/resolve/{REVISAO_IMAGENS}"
TRINCAS = {0: "D00", 1: "D10", 2: "D20"}  # a categoria 3 do espelho não é buraco e fica de fora
CLASSE_BURACO = "2"                       # em edwrdc/RDD2022: 0 line_crack, 1 group_crack, 2 pothole


def ler(url: str, tentativas: int = 10) -> bytes:
    """Baixa um arquivo e espera quando o Hugging Face pede para ir mais devagar."""
    for tentativa in range(1, tentativas + 1):
        try:
            with urllib.request.urlopen(url, timeout=60) as resposta:
                return resposta.read()
        except urllib.error.HTTPError as erro:
            if erro.code not in (429, 500, 502, 503, 504) or tentativa == tentativas:
                raise
            time.sleep(int(erro.headers.get("Retry-After") or 5 * tentativa))
        except (urllib.error.URLError, TimeoutError):
            if tentativa == tentativas:
                raise
            time.sleep(5 * tentativa)


pasta_buracos = RAIZ / "rotulos_buracos"
if not pasta_buracos.exists():
    subprocess.run(
        ["git", "clone", "-q", "--depth", "1", f"https://huggingface.co/datasets/{REPO_BURACOS}", str(pasta_buracos)],
        check=True,
        env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
    )
versao = subprocess.run(["git", "-C", str(pasta_buracos), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
if versao and versao != REVISAO_BURACOS:
    print(f"Aviso: {REPO_BURACOS} mudou. Esperado {REVISAO_BURACOS}, obtido {versao}.")

buracos = {}
for arquivo in (pasta_buracos / "labels").glob("*/*/*.txt"):
    if arquivo.parts[-3] != "test" and arquivo.parent.name in PAISES:
        linhas = [linha.split() for linha in arquivo.read_text().splitlines()]
        buracos[arquivo.stem] = [tuple(map(float, l[1:5])) for l in linhas if l[:1] == [CLASSE_BURACO]]

info = json.loads(ler(f"https://huggingface.co/api/datasets/{REPO_IMAGENS}/revision/{REVISAO_IMAGENS}"))
arquivos = [s["rfilename"] for s in info["siblings"]]
caminhos = {Path(a).stem: a for a in arquivos if a.endswith(".jpg") and "/images/" in a}
trincas = {}
for arquivo in (a for a in arquivos if a.endswith("metadata.jsonl")):
    for linha in ler(f"{BASE_IMAGENS}/{arquivo}").decode("utf-8").splitlines():
        item = json.loads(linha)
        objetos = item["objects"]
        trincas[Path(item["file_name"]).stem] = [
            (TRINCAS[c], *caixa) for caixa, c in zip(objetos["bbox"], objetos["categories"]) if c in TRINCAS
        ]

por_pais = defaultdict(list)
for nome in sorted(buracos.keys() & trincas.keys() & caminhos.keys()):
    pais = re.sub(r"_\d+$", "", nome)
    caixas = [(preparar_dados.ID_RACHADURA, 0.0, 0.0, 0.0, 0.0)] * len(trincas[nome])
    caixas += [(preparar_dados.ID_BURACO, 0.0, 0.0, 0.0, 0.0)] * len(buracos[nome])
    por_pais[pais].append(preparar_dados.Amostra(nome, pais, Path(nome), caixas))
selecionadas = [a for pais in PAISES for a in preparar_dados.selecionar(por_pais[pais], LIMITE_POR_PAIS, FRACAO_FUNDO)]
print({pais: len(lista) for pais, lista in por_pais.items()}, "selecionadas:", len(selecionadas))


def baixar_amostra(amostra) -> None:
    """Grava imagem e XML no layout do RDD2022, para o preparo seguir igual ao da base oficial."""
    pasta = ORIGEM / amostra.pais / "train"
    xml = pasta / "annotations" / "xmls" / f"{amostra.nome}.xml"
    if xml.exists():
        return
    bruto = ler(f"{BASE_IMAGENS}/{caminhos[amostra.nome]}")
    altura, largura = cv2.imdecode(np.frombuffer(bruto, np.uint8), cv2.IMREAD_COLOR).shape[:2]
    objetos = [(r, x, y, x + l, y + a) for r, x, y, l, a in trincas[amostra.nome]]
    objetos += [
        ("D40", (cx - l / 2) * largura, (cy - a / 2) * altura, (cx + l / 2) * largura, (cy + a / 2) * altura)
        for cx, cy, l, a in buracos[amostra.nome]
    ]
    corpo = "".join(
        f"<object><name>{r}</name><bndbox><xmin>{x1}</xmin><ymin>{y1}</ymin><xmax>{x2}</xmax><ymax>{y2}</ymax></bndbox></object>"
        for r, x1, y1, x2, y2 in objetos
    )
    (pasta / "images").mkdir(parents=True, exist_ok=True)
    xml.parent.mkdir(parents=True, exist_ok=True)
    (pasta / "images" / f"{amostra.nome}.jpg").write_bytes(bruto)
    xml.write_text(
        f"<annotation><size><width>{largura}</width><height>{altura}</height></size>{corpo}</annotation>",
        encoding="utf-8",
    )


inicio = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=12) as executor:
    for feitas, _ in enumerate(executor.map(baixar_amostra, selecionadas), 1):
        if feitas % 500 == 0:
            print(f"{feitas} de {len(selecionadas)} imagens em {time.time() - inicio:.0f} s", flush=True)
print(f"Base pronta em {time.time() - inicio:.0f} s")

## 5. Converter para YOLO e dividir em treino, validação e teste

In [ ]:
import json

import preparar_dados

estatisticas = preparar_dados.preparar(ORIGEM, DESTINO, LIMITE_POR_PAIS, FRACAO_FUNDO)
print(f"{'partição':<10}{'imagens':>9}{'buraco':>9}{'rachadura':>11}{'fundo':>7}")
for nome, linha in estatisticas["particoes"].items():
    print(f"{nome:<10}{linha['imagens']:>9}{linha['buraco']:>9}{linha['rachadura']:>11}{linha['fundo']:>7}")
print("Descartes:", estatisticas["descartes"] or "nenhum")
print("Rótulos ignorados:", estatisticas["rotulos_ignorados"] or "nenhum")

for classe in preparar_dados.CLASSES:
    if estatisticas["particoes"]["test"][classe] < 30:
        raise RuntimeError(f"Poucos exemplos de {classe} no teste. A avaliação não seria confiável.")

Confira se as caixas caem em cima dos defeitos. Vermelho é buraco, laranja é rachadura.

In [ ]:
import random

import cv2
import matplotlib.pyplot as plt

imagens = sorted((DESTINO / "images" / "train").glob("*.jpg"))
random.Random(SEMENTE).shuffle(imagens)
com_rotulo = [p for p in imagens if (DESTINO / "labels" / "train" / f"{p.stem}.txt").stat().st_size > 0][:6]

figura, eixos = plt.subplots(2, 3, figsize=(14, 9))
for eixo, caminho in zip(eixos.flat, com_rotulo):
    quadro = cv2.cvtColor(cv2.imread(str(caminho)), cv2.COLOR_BGR2RGB)
    altura, largura = quadro.shape[:2]
    for linha in (DESTINO / "labels" / "train" / f"{caminho.stem}.txt").read_text().splitlines():
        classe, cx, cy, l, a = map(float, linha.split())
        p1 = (int((cx - l / 2) * largura), int((cy - a / 2) * altura))
        p2 = (int((cx + l / 2) * largura), int((cy + a / 2) * altura))
        cv2.rectangle(quadro, p1, p2, (235, 40, 40) if classe == 0 else (255, 170, 0), 3)
    eixo.imshow(quadro)
    eixo.set_title(caminho.stem)
    eixo.axis("off")
plt.tight_layout()
plt.show()

## 6. Treinar

É a etapa mais longa. O mAP aparece no fim de cada época.

In [ ]:
from ultralytics import YOLO

inicio_treino = time.time()
modelo = YOLO(MODELO_BASE)
modelo.train(
    data=str(DESTINO / "dados.yaml"),
    epochs=EPOCAS,
    patience=PACIENCIA,
    imgsz=TAMANHO,
    batch=LOTE,
    workers=2,
    seed=SEMENTE,
    cos_lr=True,
    close_mosaic=10,
    flipud=0.0,   # a rua nunca aparece de cabeça para baixo
    fliplr=0.5,
    degrees=0.0,
    project=str(EXECUCOES),
    name="treino",
    exist_ok=True,
)
tempo_treino_min = (time.time() - inicio_treino) / 60
melhor_pesos = Path(modelo.trainer.best)
print(f"Treino concluído em {tempo_treino_min:.1f} min. Pesos: {melhor_pesos}")

## 7. Avaliar

Os limiares de confiança saem da **validação**, no ponto de maior F1 de cada classe. O **teste**
só é usado depois, para não contaminar a medida. A avaliação por país mostra se o modelo
funciona pior em algum lugar.

In [ ]:
import numpy as np

avaliador = YOLO(str(melhor_pesos))
OPCOES_VAL = dict(imgsz=TAMANHO, batch=LOTE, workers=2, project=str(EXECUCOES), exist_ok=True, verbose=False)


def resumir_geral(metricas) -> dict:
    caixa = metricas.box
    return {
        "precisao": float(caixa.mp),
        "revocacao": float(caixa.mr),
        "map50": float(caixa.map50),
        "map50_95": float(caixa.map),
    }


def resumir_classes(metricas, nomes: dict, limiares: dict | None = None) -> dict:
    caixa = metricas.box
    saida = {}
    for indice, classe_id in enumerate(caixa.ap_class_index):
        nome = nomes[int(classe_id)]
        precisao, revocacao, ap50, ap = caixa.class_result(indice)
        linha = {
            "instancias": int(metricas.nt_per_class[int(classe_id)]),
            "precisao": float(precisao),
            "revocacao": float(revocacao),
            "ap50": float(ap50),
            "ap50_95": float(ap),
        }
        if limiares:
            ponto = int(np.searchsorted(caixa.px, limiares[nome]))
            ponto = min(ponto, len(caixa.px) - 1)
            linha["limiar"] = limiares[nome]
            linha["precisao_no_limiar"] = float(caixa.p_curve[indice][ponto])
            linha["revocacao_no_limiar"] = float(caixa.r_curve[indice][ponto])
        saida[nome] = linha
    return saida


validacao = avaliador.val(data=str(DESTINO / "dados.yaml"), split="val", name="validacao", plots=False, **OPCOES_VAL)
NOMES = {int(k): v for k, v in validacao.names.items()}
limiares = {
    NOMES[int(classe_id)]: round(float(validacao.box.px[validacao.box.f1_curve[indice].argmax()]), 3)
    for indice, classe_id in enumerate(validacao.box.ap_class_index)
}
print("Limiares calibrados na validação:", limiares)

teste = avaliador.val(data=str(DESTINO / "dados.yaml"), split="test", name="teste", plots=True, **OPCOES_VAL)
por_pais = {}
for pasta in sorted((DESTINO / "paises").iterdir()):
    resultado = avaliador.val(
        data=str(pasta / "dados.yaml"), split="test", name=f"teste_{pasta.name}", plots=False, **OPCOES_VAL
    )
    por_pais[pasta.name] = {"geral": resumir_geral(resultado), "por_classe": resumir_classes(resultado, NOMES)}

print("Teste:", resumir_geral(teste))
for pais, dados in por_pais.items():
    print(f"  {pais}: mAP50 = {dados['geral']['map50']:.3f}")

## 8. Exportar para ONNX e conferir

O aplicativo roda o ONNX, sem PyTorch. Por isso comparo as duas versões.

In [ ]:
import detector as detector_onnx

caminho_onnx = Path(avaliador.export(format="onnx", imgsz=TAMANHO, opset=17, simplify=True, dynamic=False))
onnx_final = RAIZ / "cicloalerta.onnx"
shutil.copy2(caminho_onnx, onnx_final)

detector_app = detector_onnx.DetectorONNX(onnx_final, limiares=limiares)
imagens_teste = sorted((DESTINO / "images" / "test").glob("*.jpg"))


def iou_simples(a, b) -> float:
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    intersecao = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    uniao = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - intersecao
    return intersecao / uniao if uniao > 0 else 0.0


# Mesma imagem nas duas versões: cada caixa do PyTorch precisa existir no aplicativo.
iguais = total_caixas = 0
for caminho in imagens_teste[:100]:
    quadro = cv2.imread(str(caminho))
    resultado = avaliador.predict(quadro, conf=min(limiares.values()), iou=0.5, imgsz=TAMANHO, verbose=False)[0]
    caixas = resultado.boxes
    do_app = detector_app.detectar(quadro)
    for xyxy, confianca, classe in zip(caixas.xyxy.tolist(), caixas.conf.tolist(), caixas.cls.tolist()):
        if confianca < limiares.get(NOMES[int(classe)], 0):
            continue
        total_caixas += 1
        iguais += any(d.classe_id == int(classe) and iou_simples(d.caixa, xyxy) > 0.8 for d in do_app)
concordancia = iguais / total_caixas if total_caixas else None
print(f"Caixas iguais entre PyTorch e aplicativo: {iguais} de {total_caixas}")

latencias = []
for caminho in imagens_teste[:100]:
    quadro = cv2.imread(str(caminho))
    inicio = time.perf_counter()
    detector_app.detectar(quadro)
    latencias.append((time.perf_counter() - inicio) * 1000)
latencia_ms = float(np.median(latencias[5:])) if len(latencias) > 5 else float(np.median(latencias))
print(f"Latência em CPU do Colab: {latencia_ms:.0f} ms por quadro ({1000 / latencia_ms:.1f} FPS)")

## 9. Empacotar e baixar os resultados

In [ ]:
import csv
import platform

import torch
import ultralytics

pacote = RAIZ / "pacote"
if pacote.exists():
    shutil.rmtree(pacote)
(pacote / "modelos").mkdir(parents=True)
shutil.copy2(onnx_final, pacote / "modelos" / "cicloalerta.onnx")
shutil.copy2(melhor_pesos, pacote / "modelos" / "best.pt")
(pacote / "modelos" / "limiares.json").write_text(json.dumps(limiares, indent=2), encoding="utf-8")
shutil.copy2(DESTINO / "estatisticas.json", pacote / "estatisticas_dados.json")

for nome in ("treino", "teste"):
    pasta = pacote / nome
    pasta.mkdir()
    for arquivo in (EXECUCOES / nome).iterdir():
        if arquivo.suffix.lower() in {".png", ".jpg", ".csv", ".yaml"}:
            shutil.copy2(arquivo, pasta / arquivo.name)

# Amostras do teste para gerar os exemplos do relatório sem precisar baixar a base de novo.
com_buraco, com_rachadura, sem_defeito = [], [], []
for caminho in imagens_teste:
    rotulo = (DESTINO / "labels" / "test" / f"{caminho.stem}.txt").read_text()
    classes = {linha.split()[0] for linha in rotulo.splitlines()}
    (com_buraco if "0" in classes else com_rachadura if "1" in classes else sem_defeito).append(caminho)
for caminho in com_buraco[:30] + com_rachadura[:20] + sem_defeito[:10]:
    for subpasta, origem in (("images", caminho), ("labels", DESTINO / "labels" / "test" / f"{caminho.stem}.txt")):
        (pacote / "amostras_teste" / subpasta).mkdir(parents=True, exist_ok=True)
        shutil.copy2(origem, pacote / "amostras_teste" / subpasta / origem.name)

with (EXECUCOES / "treino" / "results.csv").open(encoding="utf-8") as arquivo:
    historico = [{k.strip(): v for k, v in linha.items()} for linha in csv.DictReader(arquivo)]
melhor_linha = max(historico, key=lambda linha: float(linha["metrics/mAP50-95(B)"]))

metricas = {
    "ambiente": {
        "python": platform.python_version(),
        "ultralytics": ultralytics.__version__,
        "torch": torch.__version__,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    },
    "configuracao": {
        "paises": PAISES,
        "limite_por_pais": LIMITE_POR_PAIS,
        "fracao_fundo": FRACAO_FUNDO,
        "epocas_maximas": EPOCAS,
        "paciencia": PACIENCIA,
        "tamanho": TAMANHO,
        "lote": LOTE,
        "modelo_base": MODELO_BASE,
        "semente": SEMENTE,
    },
    "treino": {
        "tempo_min": round(tempo_treino_min, 1),
        "epocas_executadas": len(historico),
        "melhor_epoca": int(float(melhor_linha["epoch"])),
    },
    "limiares": limiares,
    "validacao": {"geral": resumir_geral(validacao), "por_classe": resumir_classes(validacao, NOMES)},
    "teste": {
        "geral": resumir_geral(teste),
        "por_classe": resumir_classes(teste, NOMES, limiares),
        "velocidade_ms": {k: float(v) for k, v in teste.speed.items()},
    },
    "por_pais": por_pais,
    "onnx": {
        "caixas_comparadas": total_caixas,
        "concordancia_com_pytorch": concordancia,
        "latencia_cpu_colab_ms": round(latencia_ms, 1),
    },
}
(pacote / "metricas.json").write_text(json.dumps(metricas, ensure_ascii=False, indent=2), encoding="utf-8")

arquivo_final = Path(shutil.make_archive(str(RAIZ / "resultados"), "zip", pacote))
print(f"Pacote pronto: {arquivo_final} ({arquivo_final.stat().st_size / 1e6:.0f} MB)")
print(json.dumps(metricas["teste"]["geral"], indent=2))

try:
    from google.colab import files

    files.download(str(arquivo_final))
except Exception as erro:
    print(f"Baixe pelo ícone de pasta: {arquivo_final} ({erro})")

## Pronto

Envie o `resultados.zip` no chat. O relatório e os exemplos saem dele.